# Customer Churn Prediction and Customer Segmentation System

This notebook contains two major Machine Learning tasks:

1. Customer Churn Prediction
2. Customer Segmentation using K-Means Clustering

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import joblib
import warnings
warnings.filterwarnings("ignore")

sns.set(style="whitegrid")

## 2. Load Dataset

In [ ]:
df = pd.read_csv("customer_churn_data.csv")

print("Dataset Shape:", df.shape)
df.head()

## 3. Basic Data Analysis

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Churn Distribution")
plt.show()

print(df["Churn"].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="Tenure", hue="Churn", kde=True)
plt.title("Tenure Distribution by Churn")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges vs Churn")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="ContractType", hue="Churn")
plt.title("Contract Type vs Churn")
plt.show()

## 4. Churn Prediction Model

In [ ]:
X = df.drop(columns=["CustomerID", "Churn"])
y = df["Churn"]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric Features:", numeric_features)
print("Categorical Features:", categorical_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        max_depth=8
    )
}

results = {}

for model_name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    results[model_name] = {
        "pipeline": pipeline,
        "accuracy": accuracy,
        "roc_auc": roc_auc,
        "classification_report": classification_report(y_test, y_pred)
    }

    print("=" * 60)
    print(model_name)
    print("Accuracy:", round(accuracy, 4))
    print("ROC-AUC:", round(roc_auc, 4))
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

## 5. Best Model Evaluation

In [ ]:
best_model_name = max(results, key=lambda name: results[name]["roc_auc"])
best_model = results[best_model_name]["pipeline"]

print("Best Model:", best_model_name)
print("Best ROC-AUC:", round(results[best_model_name]["roc_auc"], 4))

In [ ]:
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
RocCurveDisplay.from_estimator(best_model, X_test, y_test)
plt.title(f"ROC Curve - {best_model_name}")
plt.show()

In [ ]:
joblib.dump(best_model, "churn_prediction_model.pkl")
print("Model saved as churn_prediction_model.pkl")

## 6. Customer Segmentation using K-Means

In [ ]:
segmentation_features = [
    "Age",
    "Tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SupportTickets",
    "DataUsageGB",
    "SatisfactionScore"
]

seg_df = df[segmentation_features].copy()

scaler = StandardScaler()
seg_scaled = scaler.fit_transform(seg_df)

In [ ]:
inertia = []

k_values = range(2, 9)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(seg_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(7, 5))
plt.plot(k_values, inertia, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["Segment"] = kmeans.fit_predict(seg_scaled)

segment_profile = df.groupby("Segment")[segmentation_features + ["Churn"]].mean()
segment_profile

## 7. Segment Naming

In [ ]:
segment_profile = df.groupby("Segment")[segmentation_features + ["Churn"]].mean()

highest_churn_segment = segment_profile["Churn"].idxmax()
highest_value_segment = segment_profile["TotalCharges"].idxmax()

segment_names = {}

for segment in segment_profile.index:
    if segment == highest_churn_segment:
        segment_names[segment] = "At Risk Customers"
    elif segment == highest_value_segment:
        segment_names[segment] = "High Value Customers"
    else:
        segment_names[segment] = "Regular Customers"

df["SegmentName"] = df["Segment"].map(segment_names)

print(segment_names)
df[["CustomerID", "Segment", "SegmentName"]].head()

## 8. Segment Visualization using PCA

In [ ]:
pca = PCA(n_components=2)
pca_result = pca.fit_transform(seg_scaled)

df["PCA1"] = pca_result[:, 0]
df["PCA2"] = pca_result[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="PCA1",
    y="PCA2",
    hue="SegmentName",
    palette="Set2"
)
plt.title("Customer Segments Visualization")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="SegmentName")
plt.title("Customer Count by Segment")
plt.xticks(rotation=20)
plt.show()

In [ ]:
segment_summary = df.groupby("SegmentName")[segmentation_features + ["Churn"]].mean()
segment_summary

In [ ]:
df.to_csv("segmented_customers.csv", index=False)
print("Segmented customer data saved as segmented_customers.csv")

## 9. Predict Churn for a New Customer

In [ ]:
new_customer = pd.DataFrame({
    "Gender": ["Female"],
    "Age": [32],
    "Tenure": [8],
    "MonthlyCharges": [95.50],
    "TotalCharges": [764.00],
    "ContractType": ["Month-to-month"],
    "InternetService": ["Fiber optic"],
    "PaymentMethod": ["Electronic check"],
    "SupportTickets": [5],
    "DataUsageGB": [210.50],
    "SatisfactionScore": [2]
})

prediction = best_model.predict(new_customer)[0]
probability = best_model.predict_proba(new_customer)[0][1]

print("Churn Prediction:", "Churn" if prediction == 1 else "Not Churn")
print("Churn Probability:", round(probability, 4))

## Conclusion

In this project, we successfully built:

1. A Customer Churn Prediction model
2. A Customer Segmentation system using K-Means

The churn prediction model can help businesses identify customers who may leave.  
The segmentation system helps businesses understand customer groups and create targeted marketing strategies.